# Moduł 12: Transformery

Transformery to **dominująca architektura** współczesnego deep learningu. Zastąpiły RNN/LSTM w NLP i zrewolucjonizowały wizję komputerową.

## Spis treści
1. Motywacja — dlaczego nie RNN?
2. Mechanizm atencji (Attention)
3. Scaled Dot-Product Attention
4. Multi-Head Attention
5. Architektura Transformera
6. Positional Encoding
7. Masked Attention (dekoder)
8. Vision Transformer (ViT)
9. Encoder-Decoder — architektura seq2seq
10. **Zadania**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## 1. Motywacja — dlaczego nie RNN?

| Problem RNN | Rozwiązanie Transformera |
|---|---|
| **Sekwencyjne** przetwarzanie → wolne, brak paralelizmu | **Równoległa** obróbka całej sekwencji → skaluje się na GPU |
| **Zanikający gradient** → słaba pamięć długoterminowa | **Attention** — bezpośredni dostęp do dowolnego tokenu, $O(1)$ |
| Stały bufor pamięci ($h_t$) → bottleneck | Dynamiczna atencja na **wszystkie** pozycje |
| Ciężka do skalowania | Skaluje się na tysiące GPU (GPT-4, Gemini) |

### Złożoność obliczeniowa

| Architektura | Złożoność/warstwa | Sekwencyjne operacje | Max. ścieżka |
|-------------|-------------------|---------------------|-------------|
| RNN | $O(n \cdot d^2)$ | $O(n)$ | $O(n)$ |
| CNN | $O(k \cdot n \cdot d^2)$ | $O(1)$ | $O(\log_k n)$ |
| **Transformer** | $O(n^2 \cdot d)$ | $O(1)$ | $O(1)$ |

$n$ = długość sekwencji, $d$ = wymiar modelu, $k$ = rozmiar kernela.

**Kluczowe:** Transformer ma $O(1)$ max path length (każdy token widzi każdy), ale $O(n^2)$ pamięci → problem dla długich sekwencji.

**Artykuł:** *"Attention Is All You Need"* (Vaswani et al., 2017) — 100k+ cytowań!

## 2. Mechanizm atencji — intuicja

**Atencja** to mechanizm, który pozwala modelowi "patrzeć" na różne części wejścia z **różną wagą** zależnie od kontekstu.

### Analogia — wyszukiwarka informacji

- **Query** $Q$ (zapytanie): "czego szukam?"
- **Key** $K$ (klucz): "co oferuje każdy element?"
- **Value** $V$ (wartość): "jaką informację przekazać?"

Mechanizm: Oblicz **dopasowanie** Query do każdego Key → użyj jako wag do agregacji Values.

### Wzór ogólny atencji

$$\text{Attention}(q, \{k_i, v_i\}) = \sum_{i} \alpha_i \cdot v_i, \quad \alpha_i = \frac{\exp(s(q, k_i))}{\sum_j \exp(s(q, k_j))}$$

gdzie $s(q, k)$ to **funkcja score** (dopasowanie query do key).

### Rodzaje atencji wg funkcji score

| Nazwa | Wzór $s(q, k)$ | Złożoność |
|-------|----------------|-----------|
| **Dot-product** | $q^T k$ | $O(d)$ |
| **Scaled dot-product** | $\frac{q^T k}{\sqrt{d_k}}$ | $O(d)$ |
| **Additive (Bahdanau)** | $v^T \tanh(W_1 q + W_2 k)$ | $O(d)$ |
| **General** | $q^T W k$ | $O(d^2)$ |

### Self-Attention vs Cross-Attention

- **Self-Attention:** $Q, K, V$ pochodzą z **tego samego** wejścia — każdy token "patrzy" na wszystkie inne tokeny
- **Cross-Attention:** $Q$ z jednego źródła, $K, V$ z innego (np. enkoder → dekoder w tłumaczeniu)

## 3. Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

Gdzie:
- $Q \in \mathbb{R}^{n \times d_k}$ — macierz zapytań (queries)
- $K \in \mathbb{R}^{m \times d_k}$ — macierz kluczy (keys)
- $V \in \mathbb{R}^{m \times d_v}$ — macierz wartości (values)
- $d_k$ — wymiar klucza
- Wynik: $\in \mathbb{R}^{n \times d_v}$

### Krok po kroku

1. **Score:** $S = QK^T \in \mathbb{R}^{n \times m}$ — macierz podobieństwa (iloczyn skalarny każdego query z każdym key)
2. **Scale:** $S' = \frac{S}{\sqrt{d_k}}$ — skalowanie zapobiega zbyt dużym wartościom
3. **Softmax:** $A = \text{softmax}(S') \in \mathbb{R}^{n \times m}$ — wagi atencji (sumują się do 1 po wierszach)
4. **Aggregate:** $\text{Output} = A \cdot V \in \mathbb{R}^{n \times d_v}$ — ważona suma wartości

### Dlaczego skalowanie $\frac{1}{\sqrt{d_k}}$?

Iloczyn skalarny dwóch losowych wektorów z $d_k$ wymiarami ma wariancję $\sim d_k$. Bez skalowania, dla dużych $d_k$ wartości score będą duże → softmax zwróci "spiczaste" rozkłady (prawie one-hot) → gradient $\approx 0$.

### Jak powstają Q, K, V?

Z wejścia $X \in \mathbb{R}^{n \times d}$ przez projekcje liniowe:

$$Q = X W^Q, \quad K = X W^K, \quad V = X W^V$$

gdzie $W^Q, W^K \in \mathbb{R}^{d \times d_k}$, $W^V \in \mathbb{R}^{d \times d_v}$ to **uczone** macierze wag.

In [ ]:
# Implementacja Scaled Dot-Product Attention od zera
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: Tensory (batch, seq_len, d_k / d_v)
    mask: opcjonalna maska (do dekodera)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

# Test z prostym przykładem
# 3 tokeny, d_k = 4
Q = torch.randn(1, 3, 4)
K = torch.randn(1, 3, 4)
V = torch.randn(1, 3, 4)

output, weights = scaled_dot_product_attention(Q, K, V)

print("Scores (attention weights):")
print(weights[0].detach().numpy().round(3))
print(f"\nKażdy wiersz sumuje się do: {weights[0].sum(dim=-1).detach().numpy()}")
print(f"Output shape: {output.shape}")

In [ ]:
# Wizualizacja attention weights
# Symulacja: zdanie "Kot siedzi na macie"
tokens = ["Kot", "siedzi", "na", "macie"]
n = len(tokens)

Q = torch.randn(1, n, 8)
K = torch.randn(1, n, 8)
V = torch.randn(1, n, 8)

_, weights = scaled_dot_product_attention(Q, K, V)

plt.figure(figsize=(6, 5))
plt.imshow(weights[0].detach().numpy(), cmap='Blues')
plt.xticks(range(n), tokens)
plt.yticks(range(n), tokens)
plt.xlabel("Key (na co patrzy)")
plt.ylabel("Query (kto patrzy)")
plt.colorbar(label="Waga atencji")
plt.title("Mapa atencji (losowa — przed treningiem)")

for i in range(n):
    for j in range(n):
        plt.text(j, i, f"{weights[0][i,j]:.2f}", ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 4. Multi-Head Attention

Zamiast jednej atencji, używamy **$h$ głów (heads)**, które "patrzą" na różne aspekty danych:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \cdot W^O$$

$$\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

### Wymiary

- Model: $d_{\text{model}}$ (np. 512)
- Głowy: $h$ (np. 8)
- Wymiar per głowa: $d_k = d_v = d_{\text{model}} / h$ (np. 64)
- $W_i^Q, W_i^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$, $W_i^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$
- $W^O \in \mathbb{R}^{h \cdot d_v \times d_{\text{model}}}$

### Liczba parametrów Multi-Head Attention

$$\text{params}_{\text{MHA}} = 4 \cdot d_{\text{model}}^2 + 4 \cdot d_{\text{model}} = 4d_{\text{model}}(d_{\text{model}} + 1)$$

(cztery projekcje: $W^Q, W^K, W^V, W^O$ + biasy)

Przykład: $d_{\text{model}} = 512$ → $\sim 1{,}049{,}600$ parametrów.

### Intuicja — po co wiele głów?

Każda głowa uczy się **innego typu zależności**:
- Głowa 1: zależności syntaktyczne (podmiot-orzeczenie)
- Głowa 2: koreferencja (co odnosi się do czego)
- Głowa 3: pozycyjne wzorce (sąsiednie tokeny)
- Głowa 4: semantyczne powiązania dalekiego zasięgu

Dzięki temu model jednocześnie przechwytuje różne aspekty relacji między tokenami.

In [ ]:
# Multi-Head Attention od zera
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)
        
        # Projekcja liniowa + podział na głowy
        Q = self.W_q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        # Kształt: (batch, n_heads, seq_len, d_k)
        
        # Scaled dot-product attention na każdej głowie
        attn_out, weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Konkat głów
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_k)
        
        return self.W_o(attn_out)

# Test
mha = MultiHeadAttention(d_model=32, n_heads=4)
x = torch.randn(2, 5, 32)  # batch=2, seq=5, d=32
out = mha(x, x, x)         # self-attention: Q=K=V=x
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")  # zachowuje wymiar!

## 5. Architektura Transformera

### Blok enkodera (powtórzony $N$ razy):
```
Input Embeddings + Positional Encoding
  ↓
┌───────────────────────────────────┐
│  Multi-Head Self-Attention        │
│  + Residual Connection            │
│  + Layer Normalization            │
├───────────────────────────────────┤
│  Feed-Forward Network             │
│  + Residual Connection            │
│  + Layer Normalization            │
└───────────────────────────────────┘ × N
  ↓
Output
```

### Feed-Forward Network (FFN / MLP)

Punkt-wise (niezależnie na każdym tokenie):

$$\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

- Warstwa 1: $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{ff}}$ — rozszerza wymiar
- Warstwa 2: $W_2 \in \mathbb{R}^{d_{ff} \times d_{\text{model}}}$ — kompresuje z powrotem
- Typowo: $d_{ff} = 4 \times d_{\text{model}}$ (np. 2048 gdy $d_{\text{model}}=512$)

Parametry FFN: $2 \cdot d_{\text{model}} \cdot d_{ff} + d_{ff} + d_{\text{model}}$

### Layer Normalization

W przeciwieństwie do Batch Norm, normalizuje po wymiarze **cech** (nie po batchu):

$$\text{LN}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

gdzie $\mu, \sigma^2$ liczone per token (nie per batch). Uczalne parametry: $\gamma, \beta \in \mathbb{R}^{d_{\text{model}}}$.

### Pre-Norm vs Post-Norm

- **Post-Norm** (oryginał): $\text{LN}(x + \text{Sublayer}(x))$
- **Pre-Norm** (nowsze modele, GPT): $x + \text{Sublayer}(\text{LN}(x))$ — stabilniejszy trening

### Residual Connection + Layer Norm
$$\text{output} = \text{LN}(x + \text{Sublayer}(x))$$

Residual connections umożliwiają **głębokie** sieci (jak w ResNet). Gradient przepływa bezpośrednio.

### Parametry per blok enkodera ($d_{\text{model}} = 512$, $d_{ff} = 2048$)

| Komponent | Parametry |
|-----------|-----------|
| MHA ($W^Q, W^K, W^V, W^O$) | $4 \times 512^2 + 4 \times 512 = 1{,}050{,}624$ |
| FFN ($W_1, W_2$) | $2 \times 512 \times 2048 + 2048 + 512 = 2{,}099{,}712$ |
| Layer Norm (×2) | $2 \times 2 \times 512 = 2{,}048$ |
| **Razem/blok** | **~3.15M** |
| **6 bloków** | **~18.9M** |

In [ ]:
# Blok Transformer Encoder
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-Attention + Residual + Norm
        attn_out = self.mha(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        
        # FFN + Residual + Norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        
        return x

# Test
block = TransformerEncoderBlock(d_model=32, n_heads=4, d_ff=128)
x = torch.randn(2, 10, 32)
out = block(x)
print(f"Encoder block: {x.shape} → {out.shape}")

## 6. Positional Encoding

Self-attention nie ma informacji o **pozycji** tokenów — traktuje sekwencję jak **zbiór** (permutation-invariant)!

Rozwiązanie: dodajemy **kodowanie pozycji** do embeddingów:

$$\text{Input} = \text{TokenEmbedding}(x) + \text{PositionalEncoding}(pos)$$

### Sinusoidalne kodowanie (oryginalne)

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

### Dlaczego sinusoidy?

1. Każda pozycja dostaje **unikalny wzorzec** — model rozróżnia pozycje
2. **Relatywne pozycje** — $PE_{pos+k}$ to liniowa funkcja $PE_{pos}$ (model może nauczyć się relatywnych odległości):

$$PE_{pos+k} = M_k \cdot PE_{pos}$$

dla pewnej macierzy rotacji $M_k$ zależnej tylko od $k$.

3. **Generalizacja** — model może ekstrapolować na dłuższe sekwencje niż widziane w treningu

### Alternatywy

| Typ | Opis | Model |
|-----|------|-------|
| **Sinusoidalne** | Stałe, analityczne | Oryginalny Transformer |
| **Uczone** | $PE \in \mathbb{R}^{L \times d}$ jako parametr | BERT, GPT-2 |
| **RoPE** (Rotary) | Rotacja wektorów Q/K | LLaMA, Bielik |
| **ALiBi** | Liniowy bias w attention scores | BLOOM |

### RoPE (Rotary Position Embedding) — nowoczesne

Stosuje **rotację** do wektorów Q i K:

$$f_q(x_m, m) = R_m x_m, \quad f_k(x_n, n) = R_n x_n$$

Wtedy iloczyn skalarny $f_q^T f_k = x_m^T R_{n-m} x_n$ zależy tylko od **relatywnej pozycji** $n - m$.

In [ ]:
# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)  # parzyste wymiary
        pe[:, 1::2] = torch.cos(position * div_term)  # nieparzyste wymiary
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Wizualizacja
pe = PositionalEncoding(d_model=64, max_len=100)
pe_values = pe.pe[0].numpy()

plt.figure(figsize=(12, 5))
plt.imshow(pe_values.T, aspect='auto', cmap='RdBu')
plt.xlabel("Pozycja w sekwencji")
plt.ylabel("Wymiar embeddingu")
plt.title("Positional Encoding — wzorzec sinusoidalny")
plt.colorbar()
plt.show()

## 7. Masked Attention (dekoder) — autoregresja

W **dekoderze** (generowanie, GPT) token nie może "patrzeć w przyszłość" — nie może widzieć tokenów, które jeszcze nie zostały wygenerowane.

### Causal (autoregresive) mask

Maska trójkątna dolna:

$$M_{ij} = \begin{cases}0 & \text{jeśli } j \leq i \text{ (dozwolone)} \\ -\infty & \text{jeśli } j > i \text{ (zablokowane)}\end{cases}$$

```
 1  -∞  -∞  -∞     →  softmax →    1.0   0    0    0
 1   1  -∞  -∞                      0.5  0.5   0    0
 1   1   1  -∞                      0.33 0.33 0.33  0
 1   1   1   1                      0.25 0.25 0.25 0.25
```

### Encoder vs Decoder vs Encoder-Decoder

| Architektura | Attention | Przykład |
|-------------|-----------|---------|
| **Encoder-only** | Bidirectional self-attention | BERT, RoBERTa |
| **Decoder-only** | Causal (masked) self-attention | GPT, LLaMA, Bielik |
| **Encoder-Decoder** | Encoder: bi-dir, Decoder: causal + cross-attn | T5, BART, MarianMT |

### Autoregresyjne generowanie

Model generuje tokeny **jeden po drugim**:

$$P(y_1, y_2, \ldots, y_T) = \prod_{t=1}^{T} P(y_t | y_1, \ldots, y_{t-1})$$

Na każdym kroku: cała sekwencja dotychczasowa → attention → predykcja następnego tokenu.

### Strategie dekodowania

| Strategia | Wzór | Efekt |
|-----------|------|-------|
| **Greedy** | $y_t = \arg\max P(y_t | \ldots)$ | Deterministyczne, powtarzalne |
| **Top-k sampling** | Losuj z $k$ najprawdopodobniejszych | Kreatywne, kontrolowalny losowość |
| **Top-p (nucleus)** | Losuj z najmniejszego zbioru z $\sum P \geq p$ | Bardziej adaptacyjne niż top-k |
| **Temperature** | $P'_i = \frac{e^{z_i / \tau}}{\sum_j e^{z_j / \tau}}$ | $\tau < 1$: pewniejsze, $\tau > 1$: losowe |
| **Beam search** | Utrzymuj $B$ najlepszych sekwencji | Optymalizacja globalna |

In [ ]:
# Causal mask
seq_len = 5
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)  # dolno-trójkątna
print("Causal mask:")
print(causal_mask[0])

# Attention z maską
Q = torch.randn(1, seq_len, 8)
K = torch.randn(1, seq_len, 8)
V = torch.randn(1, seq_len, 8)

out_masked, weights_masked = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

plt.figure(figsize=(6, 5))
plt.imshow(weights_masked[0].detach().numpy(), cmap='Blues')
plt.title("Masked attention weights (causal)")
plt.xlabel("Key")
plt.ylabel("Query")
plt.colorbar()
for i in range(seq_len):
    for j in range(seq_len):
        plt.text(j, i, f"{weights_masked[0][i,j]:.2f}", ha='center', va='center', fontsize=9)
plt.show()

## 8. Vision Transformer (ViT)

Transformery działają nie tylko na tekście! **ViT** (Dosovitskiy et al., 2020) pokazał, że czysty Transformer może osiągnąć state-of-the-art na ImageNet.

### Algorytm ViT

1. **Podziel** obraz $H \times W$ na **patche** $P \times P$ → $N = \frac{HW}{P^2}$ patchy
2. **Spłaszcz** każdy patch: $P \times P \times C \to P^2 C$
3. **Projekcja liniowa** na $d_{\text{model}}$: $E \in \mathbb{R}^{(P^2 C) \times d_{\text{model}}}$
4. Dodaj **[CLS] token** + **positional embedding** (uczone)
5. Przetwórz przez $L$ bloków **Transformer Encoder**
6. Użyj wyjścia **[CLS] tokenu** do klasyfikacji

### Wzory

$$\mathbf{z}_0 = [\mathbf{x}_{\text{cls}}; \; \mathbf{x}_1 E; \; \mathbf{x}_2 E; \; \ldots; \; \mathbf{x}_N E] + \mathbf{E}_{pos}$$

$$\mathbf{z}_l = \text{TransformerBlock}(\mathbf{z}_{l-1}), \quad l = 1, \ldots, L$$

$$\hat{y} = \text{MLP}(\text{LN}(\mathbf{z}_L^0))$$

```
Obraz 224×224, patch 16×16
  ↓ podziel na patche
196 patchy × (16×16×3 = 768)
  ↓ projekcja liniowa
196 tokenów × d_model
  ↓ + [CLS] + positional encoding
197 tokenów × d_model
  ↓ Transformer Encoder × L
[CLS] → MLP head → klasyfikacja
```

### Wymagania danych

ViT potrzebuje **dużo danych** (lub pretreningu na ImageNet-21k/JFT-300M). Na małych zbiorach CNN wygrywa (inductive bias konwolucji pomaga).

### Warianty

| Model | Parametry | Patch | Layers | Heads |
|-------|-----------|-------|--------|-------|
| ViT-Tiny | 5.7M | 16 | 12 | 3 |
| ViT-Small | 22M | 16 | 12 | 6 |
| ViT-Base | 86M | 16 | 12 | 12 |
| ViT-Large | 307M | 16 | 24 | 16 |
| ViT-Huge | 632M | 14 | 32 | 16 |

## 9. Encoder-Decoder — architektura seq2seq

### 9.1 Idea seq2seq

Wiele zadań wymaga mapowania **sekwencji na sekwencję** o innej długości:
- **Tłumaczenie:** "Ala ma kota" → "Alice has a cat" (4→5 tokenów)
- **Streszczanie:** długi tekst → krótkie podsumowanie
- **Question Answering:** kontekst + pytanie → odpowiedź
- **Image Captioning:** obraz → opis tekstowy

### 9.2 Architektura Encoder-Decoder Transformera

Oryginalny Transformer (Vaswani et al., 2017) to **encoder-decoder**:

```
Źródło: "Ala ma kota"                   Cel: "<sos> Alice has a cat"
       ↓                                       ↓
┌─────────────────┐                    ┌──────────────────┐
│   ENCODER       │                    │   DECODER        │
│                 │                    │                  │
│  Self-Attention │──── K, V ────────→ │  Masked Self-Att │
│  (bidirectional)│                    │        ↓         │
│        ↓        │                    │  Cross-Attention  │
│     FFN         │                    │  (Q=decoder,      │
│        ↓        │                    │   K,V=encoder)    │
│  × N layers     │                    │        ↓         │
└─────────────────┘                    │     FFN          │
                                       │  × N layers      │
                                       └──────────────────┘
                                              ↓
                                       "Alice has a cat <eos>"
```

### 9.3 Cross-Attention (kluczowy mechanizm)

W bloku dekodera występuje **cross-attention** — łączy informację z encodera:

$$\text{CrossAttention}(Q, K, V) = \text{softmax}\left(\frac{Q_{\text{dec}} K_{\text{enc}}^T}{\sqrt{d_k}}\right) V_{\text{enc}}$$

- $Q$ pochodzi od **dekodera** (bieżące tokeny wyjściowe)
- $K, V$ pochodzą od **encodera** (reprezentacja wejścia)
- Decoder "patrzy" na źródło, by zdecydować co wygenerować

### 9.4 Porównanie typów architektur

| | Encoder-only | Decoder-only | Encoder-Decoder |
|--|:---:|:---:|:---:|
| **Attention** | Bidirectional | Causal (masked) | Encoder: bidirect., Decoder: causal + cross |
| **Pretraining** | MLM | Next Token | Span corruption / denoising |
| **Mocna strona** | Rozumienie | Generowanie | Seq2Seq mapping |
| **Przykłady** | BERT, RoBERTa | GPT, LLaMA, Bielik | T5, BART, MarianMT, mBART |
| **Zadania** | Klasyfikacja, NER | Chatbot, QA, code | Tłumaczenie, streszczanie |

### 9.5 Modele Encoder-Decoder

**T5 (Text-to-Text Transfer Transformer):**
- Wszystko jako text-to-text: `"translate English to French: Hello"` → `"Bonjour"`
- Span corruption pretraining: maskuj ciągłe fragmenty → odtwórz
- Warianty: T5-Small (60M) → T5-XXL (11B)

**BART (Bidirectional and Auto-Regressive Transformer):**
- Denoising autoencoder: zaszum wejście → odtwórz oryginał
- Szumy: maskowanie tokenów, usuwanie, przestawianie, zamiana zdań
- Dobry do streszczania i generowania

**MarianMT:**
- Specjalizowany w tłumaczeniu maszynowym
- Modele per para językowa (np. `Helsinki-NLP/opus-mt-pl-en`)
- Lekkie (~300M), szybkie, open-source

### 9.6 Teacher Forcing

Podczas treningu encoder-decodera używamy **teacher forcing**:
- Zamiast karmić dekoder jego własnymi predykcjami, podajemy **ground truth** tokeny
- Przyspiesza konwergencję, ale może powodować **exposure bias** (rozbieżność trening vs inferencja)

$$\text{Train: } y_t = f(y_1^*, y_2^*, \ldots, y_{t-1}^*, \text{enc\_out})$$
$$\text{Inference: } y_t = f(\hat{y}_1, \hat{y}_2, \ldots, \hat{y}_{t-1}, \text{enc\_out})$$

### 9.7 Beam Search (powtórka z sekcji 7)

Dekodowanie w seq2seq — zamiast greedy (1 token), utrzymuj $k$ najlepszych hipotez:

$$\text{score}(\mathbf{y}) = \frac{1}{|\mathbf{y}|^{\alpha}} \sum_{t=1}^{|\mathbf{y}|} \log P(y_t | y_{<t})$$

- $k$ = beam width (typowo 4-8)
- $\alpha$ = length penalty (0.6-1.0)
- Nadal nie gwarantuje optimum, ale lepsze od greedy

In [ ]:
# =========================================
# Encoder-Decoder: Mini seq2seq w PyTorch
# =========================================
import torch
import torch.nn as nn
import numpy as np

class MiniSeq2Seq(nn.Module):
    """Uproszczony Transformer encoder-decoder do tłumaczenia."""
    def __init__(self, src_vocab, tgt_vocab, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.d_model = d_model
        self.src_embed = nn.Embedding(src_vocab, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab, d_model)
        self.pos_enc = nn.Embedding(100, d_model)  # uczone positional encoding
        
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=128,
            batch_first=True
        )
        self.fc_out = nn.Linear(d_model, tgt_vocab)
    
    def forward(self, src, tgt, tgt_mask=None):
        src_pos = torch.arange(src.size(1), device=src.device).unsqueeze(0)
        tgt_pos = torch.arange(tgt.size(1), device=tgt.device).unsqueeze(0)
        
        src_emb = self.src_embed(src) + self.pos_enc(src_pos)
        tgt_emb = self.tgt_embed(tgt) + self.pos_enc(tgt_pos)
        
        out = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask)
        return self.fc_out(out)

# Demo: Architektura
src_vocab, tgt_vocab = 100, 120
model = MiniSeq2Seq(src_vocab, tgt_vocab)
total_params = sum(p.numel() for p in model.parameters())
print(f"MiniSeq2Seq: {total_params:,} parametrów")
print(f"  Encoder layers: 2, Decoder layers: 2")
print(f"  d_model: 64, nhead: 4")

# Dummy forward pass
src = torch.randint(0, src_vocab, (2, 10))   # batch=2, src_len=10
tgt = torch.randint(0, tgt_vocab, (2, 8))    # batch=2, tgt_len=8
tgt_mask = nn.Transformer.generate_square_subsequent_mask(8)
output = model(src, tgt, tgt_mask=tgt_mask)
print(f"\n  src shape: {src.shape}")
print(f"  tgt shape: {tgt.shape}")
print(f"  output shape: {output.shape}  (batch, tgt_len, tgt_vocab)")

# =========================================
# MarianMT — tłumaczenie pl→en (HuggingFace)
# =========================================
print("\n--- MarianMT (tłumaczenie PL→EN) ---")
print("Kod do uruchomienia z internetem / na GPU:")
print("""
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-pl-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

texts = ["Ala ma kota.", "Transformery zrewolucjonizowały NLP."]
inputs = tokenizer(texts, return_tensors="pt", padding=True)
translated = model.generate(**inputs)
results = tokenizer.batch_decode(translated, skip_special_tokens=True)
for src, tgt in zip(texts, results):
    print(f"  {src} → {tgt}")
""")

---
---
# 🏋️ ZADANIA

---

### Zadanie 1: Self-Attention na zdaniu (łatwe)

1. Weź zdanie: `"Ala ma kota a kot ma Ale"`
2. Tokenizuj (split na spacje), zbuduj słownik, zamień na indeksy
3. Stwórz losowe embeddingi (`nn.Embedding`)
4. Ręcznie policz **scaled dot-product attention** (Q=K=V=embeddingi)
5. Wizualizuj mapę atencji (heatmap z tokenami na osiach)

In [ ]:
# Zadanie 1
# TWÓJ KOD TUTAJ

### Zadanie 2: Klasyfikator tekstu z TransformerEncoder (trudne)

Zbuduj klasyfikator sentymentu używając `nn.TransformerEncoder` (PyTorch):

1. Stwórz mini-dataset (20+ zdań: pozytywnych i negatywnych)
2. Model: Embedding → PositionalEncoding → TransformerEncoder → [CLS] → Linear
3. Podpowiedź: użyj `nn.TransformerEncoderLayer` i `nn.TransformerEncoder`
4. Trenuj i oceń accuracy

In [ ]:
# Zadanie 2
# TWÓJ KOD TUTAJ

### Zadanie 3: Policz parametry (średnie)

Oblicz liczbę parametrów w Transformerze:
- $d_{model} = 512$, $n_{heads} = 8$, $d_{ff} = 2048$, $n_{layers} = 6$

Dla **jednego bloku enkodera** policz parametry:
1. Multi-Head Attention: $W_Q, W_K, W_V, W_O$ (macierze + biasy)
2. Feed-Forward: $W_1, W_2$ (macierze + biasy)
3. Layer Norm: $\gamma, \beta$ (parametry per wymiar)
4. Pomnóż przez 6 warstw

Zweryfikuj wyniki tworząc model w PyTorch i zliczając `model.parameters()`.

In [ ]:
# Zadanie 3
# TWÓJ KOD TUTAJ

### Zadanie 4: Mini Vision Transformer (bonusowe)

Zaimplementuj uproszczony ViT dla MNIST (28×28):

1. Podziel obraz 28×28 na patche 7×7 → 16 patchy
2. Każdy patch: 7×7=49 → projekcja liniowa na d_model=64
3. Dodaj [CLS] token i positional encoding
4. 2 warstwy TransformerEncoder
5. [CLS] → Linear → 10 klas

Porównaj accuracy z CNN z Modułu 9.

In [ ]:
# Zadanie 4: Mini ViT
# TWÓJ KOD TUTAJ

---

## 🏆 Dodatek OAI: Ćwiczenia w stylu olimpiadowym

### Kontekst olimpiadowy
Transformery to coraz ważniejsza część OAI:

- **I OAI:** Analiza zależnościowa — structural probes na HerBERT (polski transformer)
- **I OAI, Finał:** Tłumaczenie maszynowe — implementacja atencji Bahdanau od zera
- **II OAI, Etap 2:** Ekstrakcja źródeł — SGPT-125M (transformer do embeddingów)
- **II OAI, Finał:** Stylizacja tłumaczeń — fine-tuning MarianMT
- **III OAI:** Bielik-11B-v3.0-Instruct dostępny na 2. etapie!

### Ćwiczenie OAI-12A: Embeddingi z pretrenowanego modelu

Użyj pretrenowanego modelu (np. BERT lub dowolny z HuggingFace) do uzyskania embeddingów zdań. Porównaj cosine similarity między parami zdań. To dokładnie technika z "Ekstrakcji źródeł" (II OAI).

### Ćwiczenie OAI-12B: Wizualizacja atencji

Zaimplementuj mechanizm atencji i zwizualizuj mapy atencji (heatmapa). To pojawiło się w zadaniu "Tłumaczenie maszynowe" z I OAI — podzadanie 3.

### Ćwiczenie OAI-12C: Tokenizacja i template modelu

Przygotuj się na Bielika: napisz kod, który formatuje prompt w formacie ChatML (używanym przez Bielik-11B-v3.0-Instruct) i tokenizuje go.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# === OAI-12A: Cosine similarity embeddingów ===
print("=== Embedding Similarity (wzorzec: Ekstrakcja źródeł, II OAI) ===")

# Symulacja embeddingów (bez transformera — dla demonstracji)
# Na prawdziwej olimpiadzie użyjesz model.encode() z HuggingFace
np.random.seed(42)
dim = 768  # Typowy wymiar embeddingów (BERT, SGPT, etc.)

# "Embeddingi" dokumentów
docs = {
    "doc_0": "Sieci neuronowe uczą się z danych",
    "doc_1": "Deep learning to podzbiór uczenia maszynowego",
    "doc_2": "Kot siedzi na macie",
    "doc_3": "Pies biega po ogrodzie",
    "doc_4": "Gradient descent optymalizuje parametry modelu"
}

# Symulowane embeddingi (w praktyce: model.encode(text))
embeddings = {}
base = np.random.randn(dim)
for i, (key, text) in enumerate(docs.items()):
    # Dokumenty o AI są bliżej siebie
    if "siec" in text.lower() or "learn" in text.lower() or "gradient" in text.lower():
        embeddings[key] = base + 0.1 * np.random.randn(dim)
    else:
        embeddings[key] = np.random.randn(dim)
    embeddings[key] /= np.linalg.norm(embeddings[key])

query_emb = base / np.linalg.norm(base)  # Zapytanie: coś o AI

# Ranking — cosine similarity
print(f"\n  Zapytanie: 'sztuczna inteligencja'")
similarities = {}
for key, emb in embeddings.items():
    sim = np.dot(query_emb, emb)
    similarities[key] = sim

for key, sim in sorted(similarities.items(), key=lambda x: -x[1]):
    print(f"  {key}: sim={sim:.4f} — '{docs[key]}'")

# === OAI-12B: Wizualizacja atencji ===
print("\n=== Wizualizacja atencji (wzorzec: Tłumaczenie maszynowe, I OAI) ===")

def visualize_attention(source_tokens, target_tokens, attention_weights):
    """Wizualizacja mapy atencji jak w pracy Bahdanau (I OAI finał)."""
    fig, ax = plt.subplots(figsize=(8, 6))
    cax = ax.matshow(attention_weights, cmap='YlOrRd')
    fig.colorbar(cax)
    
    ax.set_xticks(range(len(source_tokens)))
    ax.set_yticks(range(len(target_tokens)))
    ax.set_xticklabels(source_tokens, rotation=45, ha='left')
    ax.set_yticklabels(target_tokens)
    ax.set_xlabel("Źródło (encoder)")
    ax.set_ylabel("Cel (decoder)")
    ax.set_title("Mapa atencji — wzorzec z I OAI")
    plt.tight_layout()
    plt.show()

# Symulacja atencji (tłumaczenie DE→EN)
source = ["Ich", "bin", "ein", "Student", "."]
target = ["I", "am", "a", "student", "."]

# Idealnie: atencja powinna być diagonalna (z lekkim przesunięciem)
np.random.seed(42)
attn = np.eye(5) * 0.6 + 0.1 * np.random.rand(5, 5)
attn = attn / attn.sum(axis=1, keepdims=True)  # Normalizacja (softmax)

visualize_attention(source, target, attn)

# === OAI-12C: ChatML template dla Bielika ===
print("\n=== ChatML Template — Bielik-11B-v3.0-Instruct ===")

def format_chatml(messages):
    """
    Formatuj wiadomości w formacie ChatML (używanym przez Bielik).
    Każda wiadomość: <|im_start|> rola\ntreść<|im_end|>
    """
    formatted = "<s>"
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        formatted += f"<|im_start|> {role}\n{content}<|im_end|> \n"
    # Dodaj początek odpowiedzi asystenta
    formatted += "<|im_start|> assistant\n"
    return formatted

# Przykład użycia
messages = [
    {"role": "system", "content": "Jesteś ekspertem od sztucznej inteligencji. Odpowiadaj krótko po polsku."},
    {"role": "user", "content": "Wyjaśnij czym jest gradient descent w 2 zdaniach."}
]

prompt = format_chatml(messages)
print("Sformatowany prompt dla Bielika:")
print("-" * 60)
print(prompt)
print("-" * 60)

# Na prawdziwej olimpiadzie:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("speakleash/Bielik-11B-v3.0-Instruct")
# model = AutoModelForCausalLM.from_pretrained("speakleash/Bielik-11B-v3.0-Instruct", 
#                                               torch_dtype=torch.bfloat16)
# input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt")
# output = model.generate(input_ids.to("cuda"), max_new_tokens=200)
# print(tokenizer.decode(output[0]))

print("\n💡 Na 2. etapie III OAI będziesz mógł użyć Bielika — przygotuj prompte z góry!")